# Spatial relationships in Calgary's water network

How closely do historical breaks align with the current pipe inventory? How much pipe intersects each 500 m sector, and where have breaks recurred?

Run the cells in order from a HYDROSIGHT checkout with its DVC data restored. In Colab, clone the repository to `/content/HYDROSIGHT` and restore the same DVC artifacts before running. The notebook reads the project data and prints descriptive results.

In [ ]:
from pathlib import Path
import json
import sys

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import shape

candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/HYDROSIGHT")]
root = next((p for p in candidates if (p / "src/hydrosight/preprocessing/break_history.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook in the HYDROSIGHT checkout.")

sys.path.insert(0, str(root / "src"))
from hydrosight.preprocessing.break_history import assign_breaks_to_grid

data = root / "data"
history_dir = data / "pipeline/break_history"
required = [
    data / "pipeline/preprocessed/pipes.jsonl",
    data / "pipeline/preprocessed/water_breaks.jsonl",
    data / "processed/spatial_grid.geojson",
    data / "processed/spatial_grid_manifest.json",
    history_dir / "breaks_by_sector_month.csv",
    history_dir / "break_history_quality.json",
]
missing = [str(path.relative_to(root)) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Run dvc pull in the checkout to restore: " + ", ".join(missing))

In [ ]:
grid = gpd.read_file(data / "processed/spatial_grid.geojson")
df_clean = pd.read_json(data / "pipeline/preprocessed/pipes.jsonl", lines=True)
breaks = pd.read_json(data / "pipeline/preprocessed/water_breaks.jsonl", lines=True)
monthly = pd.read_csv(history_dir / "breaks_by_sector_month.csv", parse_dates=["month"])
quality = json.loads((history_dir / "break_history_quality.json").read_text())
manifest = json.loads((data / "processed/spatial_grid_manifest.json").read_text())

if quality["grid_version"] != manifest["active_version"] or len(grid) != manifest["cell_count"]:
    raise ValueError("Break history and the active sector grid must match.")
if len(breaks) != quality["source_records"] or monthly["break_count"].sum() != quality["assigned_records"]:
    raise ValueError("Break totals differ from the persisted history quality report.")

print(f"Grid version: {quality['grid_version']} | Sectors: {len(grid):,}")
print(f"Historical breaks: {len(breaks):,} | Outside grid: {quality['unassigned_records']:,}")

## 1. Historical breaks and nearby pipes

For each break, find the nearest mapped pipe within 30 m. An ambiguous match means multiple pipes tied for the shortest distance. Distances describe proximity to the current network; a unique nearest pipe alone cannot identify which pipe failed.

In [ ]:
def assign_pipes_to_grid(
    df_pipes: pd.DataFrame,
    grid: gpd.GeoDataFrame
) -> gpd.GeoDataFrame:
    """Assign pipes to spatial grid cells using their geometries."""

    pipes = gpd.GeoDataFrame(
        df_pipes.copy(),
        geometry=df_pipes["multilinestring"].apply(
            lambda value: shape(json.loads(value))
            if isinstance(value, str)
            else shape(value)
        ),
        crs="EPSG:4326"
    )

    pipes = pipes.to_crs(grid.crs)

    gdf_pipes_grid = gpd.sjoin(
        pipes,
        grid[["cell_id", "geometry"]],
        how="left",
        predicate="intersects"
    )

    return gdf_pipes_grid


def match_breaks_to_pipes(
    gdf_pipes: gpd.GeoDataFrame,
    gdf_water_breaks: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    """Match water main breaks to nearby pipes using metric distances."""

    pipes = (
        gdf_pipes
        .drop(columns=["index_right"], errors="ignore")
        .drop_duplicates(subset=["globalid"])
        .to_crs("EPSG:32611")
    )

    breaks = (
        gdf_water_breaks
        .drop(columns=["index_right"], errors="ignore")
        .reset_index(drop=True)
        .to_crs("EPSG:32611")
        .copy()
    )

    breaks["break_id"] = breaks.index

    gdf_pipe_breaks = gpd.sjoin_nearest(
        breaks,
        pipes[["globalid", "geometry"]],
        how="left",
        max_distance=30,
        distance_col="distance_m",
    )

    gdf_pipe_breaks = gdf_pipe_breaks.drop(
        columns=["index_right"],
        errors="ignore"
    )

    gdf_pipe_breaks["match"] = np.select(
        [
            gdf_pipe_breaks["globalid"].isna(),
            gdf_pipe_breaks.duplicated(
                subset=["break_id"],
                keep=False
            )
        ],
        [
            "Absent",
            "Ambiguous"
        ],
        default="Unique"
    )

    return gdf_pipe_breaks


def print_pipe_match_summary(
    gdf_pipe_breaks: gpd.GeoDataFrame,
    original_break_count: int,
) -> None:
    break_match_summary = gdf_pipe_breaks.drop_duplicates(subset=["break_id"])

    print("=== Breaks and the nearest pipes in the current network ===")
    print(f"Total breaks in the full history: {original_break_count:,}")
    print("Match status (one count per break):")
    match_counts = break_match_summary["match"].value_counts().rename_axis(None)
    print(match_counts.to_string())

    print("\nDistances to candidate pipes (m, one row per candidate):")
    print(gdf_pipe_breaks["distance_m"].describe().to_string())

In [ ]:
pipes_by_sector = assign_pipes_to_grid(df_clean, grid)
breaks_by_sector = assign_breaks_to_grid(breaks, grid)
matches = match_breaks_to_pipes(pipes_by_sector, breaks_by_sector)
print_pipe_match_summary(matches, len(breaks))

## 2. Current pipe coverage by sector

Summarize pipe–sector intersections and the length of pipe inside each sector. The project's intersection rule includes boundary touches. A sector with an intersection can therefore have zero measured pipe length.

In [ ]:
def aggregate_pipe_features(
    gdf_pipes_grid: gpd.GeoDataFrame,
    grid: gpd.GeoDataFrame,
) -> pd.DataFrame:
    """Aggregate pipe characteristics by spatial grid cell."""

    pipes = gdf_pipes_grid.dropna(subset=["cell_id"]).copy()

    pipes = pipes.to_crs("EPSG:32611")
    cells = grid.to_crs("EPSG:32611")

    cell_geometries = cells.set_index("cell_id")["geometry"]

    pipes["pipe_length_in_cell_m"] = pipes.apply(
        lambda row: row.geometry.intersection(
            cell_geometries.loc[row["cell_id"]]
        ).length,
        axis=1,
    )

    df_features = (
        pipes.groupby("cell_id")
        .agg(
            pipe_count=("globalid", "nunique"),
            pipe_length_m=("pipe_length_in_cell_m", "sum"),
            avg_diameter=("diam", "mean"),
            material_count=("material", "nunique"),
        )
        .reset_index()
    )

    return df_features


def print_pipe_coverage_summary(
    gdf_pipes_grid: gpd.GeoDataFrame,
    df_pipe_features: pd.DataFrame,
) -> None:
    duplicate_pairs = gdf_pipes_grid.duplicated(subset=["globalid", "cell_id"]).sum()
    total_length_m = round(df_pipe_features["pipe_length_m"].sum(), 2)

    print("\n=== Current pipe network by sector ===")
    print(f"Sectors with pipes: {len(df_pipe_features):,}")
    print(f"Total network length in the grid: {total_length_m:,.2f} m")
    print(f"Duplicate pipe–sector pairs: {duplicate_pairs:,}")

    print("\nSample of pipe features by sector:")
    print(df_pipe_features.head().to_string(index=False))

In [ ]:
pipe_features = aggregate_pipe_features(pipes_by_sector, grid)
print_pipe_coverage_summary(pipes_by_sector, pipe_features)
print("Sectors with positive pipe length:", pipe_features["pipe_length_m"].gt(0).sum())

## 3. Recorded break recurrence and network overlap, 2016–2025

Use the persisted monthly sector history for the ten complete calendar years in this comparison. Compare sectors with observed breaks against sectors intersecting the current pipe inventory. A sector with no recorded break may have incomplete observation coverage. The current inventory also may differ from the network present when the break occurred.

In [ ]:
def print_break_coverage_and_recurrence(
    df_break_history: pd.DataFrame,
    breaks_in_period: pd.DataFrame,
    pipe_cells: set[str],
) -> None:
    break_cells = set(breaks_in_period["cell_id"].dropna())

    print("\n=== Recorded breaks by sector ===")
    print(f"Breaks assigned to the grid, all years: {df_break_history['break_count'].sum():,}")
    print("\nSample of monthly break counts from the full history:")
    print(df_break_history.head().to_string(index=False))

    print("\nBreaks in 2016–2025 and overlap with the current pipe network:")
    print(f"  Recorded breaks: {breaks_in_period['break_count'].sum():,}")
    print(f"  Sectors with recorded breaks: {len(break_cells):,}")
    print(f"  Sectors with current pipes and recorded breaks: {len(pipe_cells & break_cells):,}")
    print(f"  Sectors with recorded breaks and no current pipes: {len(break_cells - pipe_cells):,}")
    print(f"  Sectors with current pipes and no recorded breaks: {len(pipe_cells - break_cells):,}")

    breaks_per_cell = breaks_in_period.groupby("cell_id")["break_count"].sum()

    print("\nBreak recurrence by sector (2016–2025):")
    print(f"  Exactly one break: {breaks_per_cell.eq(1).sum():,}")
    print(f"  Two or more breaks: {breaks_per_cell.ge(2).sum():,}")
    print(f"  Highest break count in one sector: {breaks_per_cell.max():,}")

In [ ]:
study_months = monthly.loc[monthly["month"].dt.year.between(2016, 2025)]
pipe_cells = set(pipe_features["cell_id"].dropna())
print_break_coverage_and_recurrence(monthly, study_months, pipe_cells)